<a href="https://colab.research.google.com/github/mcarden4-netizen/Homework-Intro-to-ML/blob/Homework/Homework5part3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import torch
from google.colab import drive
drive.mount('/content/drive')

filetest = '/content/drive/My Drive/Housing.csv'


torch.set_printoptions(edgeitems=2)

data = pd.read_csv('/content/drive/My Drive/Housing.csv')
data.head()

Mounted at /content/drive


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [ ]:
X = data.drop(columns=['price'])
X = pd.get_dummies(X, drop_first=True, dtype=float)
X = torch.tensor(X.values, dtype=torch.float32)

Y = torch.tensor(data['price'].values, dtype=torch.float32)

Y = Y / 1e6

In [ ]:
torch.manual_seed(50)

Ts = X.shape[0]
Vs = int(0.2 * Ts)

shuffled_indices = torch.randperm(Ts)

traindata = shuffled_indices[:-Vs]
valdata = shuffled_indices[-Vs:]

Xtrain = X[traindata]
Xval = X[valdata]

Ytrain = Y[traindata]
Yval = Y[valdata]

In [ ]:
Xtrainm = Xtrain.mean(dim=0)
Xtrainstd = Xtrain.std(dim=0)

train_X = (Xtrain - Xtrainm) / Xtrainstd
val_X = (Xval - Xtrainm) / Xtrainstd

In [ ]:
def model(X, params):
    return X @ params[:-1] + params[-1]

def loss_fn(YP, Y):
    squared_diffs = (YP - Y) ** 2
    return squared_diffs.mean()


In [ ]:
def training_loop(n_epochs, learning_rate, params, Xtrain, Ytrain, Xval, Yval):

    for epoch in range(1, n_epochs + 1):

        if params.grad is not None:
            params.grad.zero_()

        YPtrain = model(Xtrain, params)

        train_loss = loss_fn(YPtrain, Ytrain)

        train_loss.backward()

        with torch.no_grad():
            params -= learning_rate * params.grad

        with torch.no_grad():

            YPval = model(Xval, params)

            val_loss = loss_fn(YPval, Yval)

        if epoch % 500 == 0:
            print('Epoch %d, Training Loss %f, Validation Loss %f'% (epoch, float(train_loss), float(val_loss))
            )

    return params


In [ ]:
learning_rates = [1e-1, 1e-2, 1e-3, 1e-4]


for learning_rate in learning_rates:

    print("\nLearning rate:", learning_rate)

    number_of_features = train_X.shape[1]

    params = torch.tensor(
        [1.0] + [0.0] * number_of_features,
        requires_grad=True
    )

    params = training_loop(n_epochs=5000, learning_rate=learning_rate, params=params, Xtrain=train_X, Ytrain=Ytrain, Xval=val_X, Yval=Yval)

    print("Final parameters:")
    print(params)

    with torch.no_grad():

        val_predictions = model(val_X, params)

        final_val_loss = loss_fn(val_predictions, Yval)

    print("Final validation loss:", float(final_val_loss))


Learning rate: 0.1


/tmp/ipykernel_1924/1378649810.py:24: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Epoch %d, Training Loss %f, Validation Loss %f'% (epoch, float(train_loss), float(val_loss))


Epoch 500, Training Loss 1.070433, Validation Loss 1.297158
Epoch 1000, Training Loss 1.070433, Validation Loss 1.297158
Epoch 1500, Training Loss 1.070433, Validation Loss 1.297158
Epoch 2000, Training Loss 1.070433, Validation Loss 1.297158
Epoch 2500, Training Loss 1.070433, Validation Loss 1.297158
Epoch 3000, Training Loss 1.070433, Validation Loss 1.297158
Epoch 3500, Training Loss 1.070433, Validation Loss 1.297158
Epoch 4000, Training Loss 1.070433, Validation Loss 1.297158
Epoch 4500, Training Loss 1.070433, Validation Loss 1.297158
Epoch 5000, Training Loss 1.070433, Validation Loss 1.297158
Final parameters:
tensor([ 0.5433,  0.0941,  0.5078,  0.3759,  0.2127,  0.1767,  0.1032,  0.1706,
         0.1865,  0.4127,  0.2607,  0.0168, -0.1963,  4.7924],
       requires_grad=True)
Final validation loss: 1.2971582412719727

Learning rate: 0.01
Epoch 500, Training Loss 1.070437, Validation Loss 1.297457
Epoch 1000, Training Loss 1.070433, Validation Loss 1.297159
Epoch 1500, Trainin